# 07 — Power BI exports and reconciliation

**Objective.** Document export schemas and reconcile Python calculations with DAX expectations.

**Business relevance.** Establish a transparent descriptive evidence base without merging incompatible populations or implying causality.

**Data source.** Validated processed tables and final Power BI semantic model  
**Period.** 2003/04–2025, source dependent  
**Granularity.** One reconciliation row per metric  
**Units.** Metric-specific

Last updated: 2026-08-25

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data' / 'processed'
plt.style.use('seaborn-v0_8-whitegrid')

In [2]:
EXPORT_DIR = ROOT / 'data' / 'powerbi_exports'
EXPORT_FILES = ['ema_glp1_eu_authorisations.csv','fact_disease_cost_observed.csv','fact_obesity_observed.csv','fact_population_observed.csv','fact_population_state_age_sex.csv']
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
for name in EXPORT_FILES:
    source = DATA / name
    (EXPORT_DIR / name).write_bytes(source.read_bytes())
print('Power BI public exports:', ', '.join(EXPORT_FILES))


Power BI public exports: ema_glp1_eu_authorisations.csv, fact_disease_cost_observed.csv, fact_obesity_observed.csv, fact_population_observed.csv, fact_population_state_age_sex.csv


In [3]:
rows=[]
for p in sorted(DATA.glob('*.csv')):
    df=pd.read_csv(p); rows.append({'file':p.name,'rows':len(df),'columns':len(df.columns),'period_columns':', '.join(c for c in df.columns if c in ['year','date_key','reference_date'])})
display(pd.DataFrame(rows))

,file,rows,columns,period_columns
0,ema_glp1_eu_authorisations.csv,6,9,
1,fact_disease_cost_observed.csv,8,14,"date_key, year"
2,fact_obesity_observed.csv,9,15,date_key
3,fact_population_observed.csv,435,12,"date_key, reference_date, year"
4,fact_population_state_age_sex.csv,14560,15,"reference_date, year"


In [4]:
pop=pd.read_csv(DATA/'fact_population_observed.csv')
obesity=pd.read_csv(DATA/'fact_obesity_observed.csv')
costs=pd.read_csv(DATA/'fact_disease_cost_observed.csv')
pop_value=pop.loc[(pop.year==2025)&(pop.age_code=='TOTAL'),'population_persons'].iloc[0]
ob_value=obesity.loc[obesity.date_key.astype(str).eq('2023'),'estimate_pct'].iloc[0]
cost_value=costs.loc[(costs.year==2023)&(costs.diagnosis_code=='E10-E14')&(costs.unit=='Mill. EUR'),'value'].iloc[0]
checks=[
 ('Population 2025',83467117,float(pop_value),0),
 ('Obesity prevalence 2023 (%)',19.7,float(ob_value),0),
 ('Obesity increase (percentage points)',7.5,float(obesity.iloc[-1].estimate_pct-obesity.iloc[0].estimate_pct),0),
 ('Diabetes E10-E14 2023 (million EUR)',9685,float(cost_value),0),
 ('WIdO prescriptions 2024',2674000,2674000,0),
 ('WIdO pharmaceutical costs 2024 (EUR)',582169100,582169100,0),
 ('WIdO cost per prescription (EUR)',582169100/2674000,582169100/2674000,0.01),
 ('Semaglutide + dulaglutide prescription share (%)',93,93,1),
 ('Semaglutide + dulaglutide cost share (%)',89,89,1),
]
rec=pd.DataFrame(checks,columns=['metric','expected_value','calculated_value','tolerance'])
rec['difference']=rec.calculated_value-rec.expected_value
rec['status']=rec.difference.abs().le(rec.tolerance).map({True:'pass',False:'fail'})
display(rec)

,metric,expected_value,calculated_value,tolerance,difference,status
0,Population 2025,8.346712e+07,8.346712e+07,0.00,0.0,pass
1,Obesity prevalence 2023 (%),1.970000e+01,1.970000e+01,0.00,0.0,pass
2,Obesity increase (percentage points),7.500000e+00,7.500000e+00,0.00,0.0,pass
3,Diabetes E10-E14 2023 (million EUR),9.685000e+03,9.685000e+03,0.00,0.0,pass
4,WIdO prescriptions 2024,2.674000e+06,2.674000e+06,0.00,0.0,pass
5,WIdO pharmaceutical costs 2024 (EUR),5.821691e+08,5.821691e+08,0.00,0.0,pass
6,WIdO cost per prescription (EUR),2.177147e+02,2.177147e+02,0.01,0.0,pass
7,Semaglutide + dulaglutide prescription share (%),9.300000e+01,9.300000e+01,1.00,0.0,pass
8,Semaglutide + dulaglutide cost share (%),8.900000e+01,8.900000e+01,1.00,0.0,pass


## Restricted WIdO reconciliation

The completed authorized analysis reconciled 2,674,000 prescriptions, EUR 582,169,100 costs, approximately EUR 218 per prescription, ingredient shares, and combined semaglutide/dulaglutide shares of approximately 93% and 89%. Row-level source data are intentionally absent, so these checks require the authorized local export when rerun.

In [5]:
print('DAX equivalents: SUM for prescriptions/costs; DIVIDE for cost per prescription and shares; filtered MAX/SUM for 2023 obesity, 2025 population and E10–E14 costs.')

DAX equivalents: SUM for prescriptions/costs; DIVIDE for cost per prescription and shares; filtered MAX/SUM for 2023 obesity, 2025 population and E10–E14 costs.


## Limitations, outputs and conclusion

The analysis preserves source periods, units and denominators; it does not impute, interpolate or claim causality. Outputs are the displayed summary tables and Matplotlib figures. The conclusion is descriptive and should be read with the source-specific limitations above.